# 📈 MACD Oscillator 백테스팅 (대화형)

**개인용 백테스팅 노트북** - 백엔드 없이 바로 실행!

## 사용 방법:
1. 셀을 순서대로 실행하세요 (`Shift + Enter`)
2. 파라미터를 수정하고 다시 실행하세요
3. 모든 데이터는 로컬에만 저장됩니다

---

In [ ]:
# 패키지 설치 (최초 1회만 실행)
# !pip install yfinance pandas numpy matplotlib plotly ipywidgets

In [ ]:
# 필수 라이브러리 임포트
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Jupyter 노트북 설정
%matplotlib inline
plt.style.use('seaborn-v0_8-darkgrid')

print("✅ 라이브러리 로드 완료!")

## 📊 1. 파라미터 설정

**여기만 수정하면 됩니다!**

In [ ]:
# ===== 여기를 수정하세요 =====

TICKER = 'AAPL'           # 티커 심볼 (예: AAPL, NVDA, TSLA, MSFT)
MA1 = 10                  # 단기 이동평균 (권장: 10)
MA2 = 21                  # 장기 이동평균 (권장: 21)
START_DATE = '2020-01-01' # 시작 날짜
END_DATE = '2024-01-01'   # 종료 날짜
INITIAL_CAPITAL = 20000   # 초기 자본 ($)

# ===========================

print(f"📌 설정:")
print(f"   티커: {TICKER}")
print(f"   이동평균: MA{MA1} / MA{MA2}")
print(f"   기간: {START_DATE} ~ {END_DATE}")
print(f"   초기 자본: ${INITIAL_CAPITAL:,}")

## 📥 2. 데이터 다운로드

Yahoo Finance에서 무료 데이터를 가져옵니다.

In [ ]:
# 데이터 다운로드
print(f"📥 {TICKER} 데이터 다운로드 중...")

df = yf.download(TICKER, start=START_DATE, end=END_DATE, progress=False)

if df.empty:
    print(f"❌ 오류: '{TICKER}'에 대한 데이터를 찾을 수 없습니다.")
else:
    print(f"✅ {len(df)}개 데이터 포인트 다운로드 완료!\n")
    print("첫 5개 행:")
    display(df.head())
    print(f"\n마지막 5개 행:")
    display(df.tail())

## ⚙️ 3. MACD 전략 실행

In [ ]:
# MACD 계산
df['MA1'] = df['Close'].rolling(window=MA1, min_periods=1).mean()
df['MA2'] = df['Close'].rolling(window=MA2, min_periods=1).mean()

# 포지션 생성
df['Position'] = 0
df.loc[MA1:, 'Position'] = np.where(df['MA1'][MA1:] >= df['MA2'][MA1:], 1, 0)

# 시그널 생성
df['Signal'] = df['Position'].diff()

# Oscillator
df['Oscillator'] = df['MA1'] - df['MA2']

# 수익률 계산
df['Returns'] = df['Close'].pct_change()
df['Strategy_Returns'] = df['Position'].shift(1) * df['Returns']

# 누적 수익률
df['Cumulative_Returns'] = (1 + df['Returns']).cumprod()
df['Cumulative_Strategy_Returns'] = (1 + df['Strategy_Returns']).cumprod()

# 포트폴리오 가치
df['Portfolio_Value'] = INITIAL_CAPITAL * df['Cumulative_Strategy_Returns']

print("✅ MACD 전략 계산 완료!")
print(f"\n매수 신호: {(df['Signal'] == 1).sum()}회")
print(f"매도 신호: {(df['Signal'] == -1).sum()}회")

## 📊 4. 성과 지표

In [ ]:
# 성과 계산
total_return = (df['Portfolio_Value'].iloc[-1] - INITIAL_CAPITAL) / INITIAL_CAPITAL * 100
buy_hold_return = (df['Cumulative_Returns'].iloc[-1] - 1) * 100

# Sharpe ratio
sharpe = (df['Strategy_Returns'].mean() / df['Strategy_Returns'].std()) * np.sqrt(252)

# Max drawdown
cumulative = df['Cumulative_Strategy_Returns']
running_max = cumulative.cummax()
drawdown = (cumulative - running_max) / running_max
max_drawdown = drawdown.min() * 100

# 승률
winning_trades = (df['Strategy_Returns'] > 0).sum()
total_trades = (df['Signal'] != 0).sum()
win_rate = (winning_trades / total_trades * 100) if total_trades > 0 else 0

# 결과 출력
print("="*60)
print(f"{'백테스팅 성과 요약':^60}")
print("="*60)
print(f"\n🎯 총 수익률:        {total_return:>10.2f}%")
print(f"📈 매수&보유 수익률:  {buy_hold_return:>10.2f}%")
print(f"💰 초과 수익:        {total_return - buy_hold_return:>10.2f}%")
print(f"\n📊 샤프 비율:        {sharpe:>10.2f}")
print(f"📉 최대 낙폭:        {max_drawdown:>10.2f}%")
print(f"🎲 승률:            {win_rate:>10.1f}%")
print(f"\n💵 초기 자본:        ${INITIAL_CAPITAL:>10,.2f}")
print(f"💵 최종 자산:        ${df['Portfolio_Value'].iloc[-1]:>10,.2f}")
print(f"💵 손익:            ${df['Portfolio_Value'].iloc[-1] - INITIAL_CAPITAL:>10,.2f}")
print(f"\n🔄 총 거래 횟수:      {total_trades:>10}회")
print("="*60)

## 📈 5. 시각화 - 대화형 차트 (Plotly)

**차트 기능:**
- 🔍 줌 인/아웃
- 🖱️ 드래그로 팬
- 💾 이미지로 저장 가능

In [ ]:
# 대화형 차트 생성
fig = make_subplots(
    rows=3, cols=1,
    shared_xaxes=True,
    vertical_spacing=0.05,
    row_heights=[0.5, 0.25, 0.25],
    subplot_titles=(f'{TICKER} - 가격 & 신호', 'MACD Oscillator', '포트폴리오 가치')
)

# 1. 캔들스틱 차트
fig.add_trace(
    go.Candlestick(
        x=df.index,
        open=df['Open'],
        high=df['High'],
        low=df['Low'],
        close=df['Close'],
        name='가격'
    ),
    row=1, col=1
)

# 이동평균선
fig.add_trace(
    go.Scatter(x=df.index, y=df['MA1'], mode='lines', name=f'MA{MA1}', line=dict(color='blue', width=1)),
    row=1, col=1
)
fig.add_trace(
    go.Scatter(x=df.index, y=df['MA2'], mode='lines', name=f'MA{MA2}', line=dict(color='orange', width=1, dash='dot')),
    row=1, col=1
)

# 매수/매도 신호
buy_signals = df[df['Signal'] == 1]
sell_signals = df[df['Signal'] == -1]

fig.add_trace(
    go.Scatter(
        x=buy_signals.index, y=buy_signals['Close'],
        mode='markers', name='매수',
        marker=dict(symbol='triangle-up', size=12, color='green')
    ),
    row=1, col=1
)

fig.add_trace(
    go.Scatter(
        x=sell_signals.index, y=sell_signals['Close'],
        mode='markers', name='매도',
        marker=dict(symbol='triangle-down', size=12, color='red')
    ),
    row=1, col=1
)

# 2. MACD Oscillator
colors = ['red' if val < 0 else 'green' for val in df['Oscillator']]
fig.add_trace(
    go.Bar(x=df.index, y=df['Oscillator'], name='MACD', marker_color=colors),
    row=2, col=1
)

# 3. 포트폴리오 가치
fig.add_trace(
    go.Scatter(
        x=df.index, y=df['Portfolio_Value'],
        mode='lines', name='포트폴리오',
        line=dict(color='purple', width=2),
        fill='tozeroy'
    ),
    row=3, col=1
)

# 레이아웃 설정
fig.update_layout(
    height=900,
    showlegend=True,
    xaxis_rangeslider_visible=False,
    hovermode='x unified',
    title_text=f'{TICKER} MACD 백테스팅 결과'
)

fig.update_yaxes(title_text="가격 ($)", row=1, col=1)
fig.update_yaxes(title_text="Oscillator", row=2, col=1)
fig.update_yaxes(title_text="포트폴리오 가치 ($)", row=3, col=1)
fig.update_xaxes(title_text="날짜", row=3, col=1)

fig.show()

print("\n💡 TIP: 차트를 드래그하여 확대/축소할 수 있습니다!")

## 📊 6. 정적 차트 (Matplotlib) - 저장용

In [ ]:
# Matplotlib 차트 (저장하기 좋음)
fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(15, 10), sharex=True)

# 1. 가격 & 신호
ax1.plot(df.index, df['Close'], label=TICKER, alpha=0.7, linewidth=2)
ax1.plot(df.index, df['MA1'], label=f'MA{MA1}', linewidth=1.5)
ax1.plot(df.index, df['MA2'], label=f'MA{MA2}', linewidth=1.5, linestyle='--')
ax1.scatter(buy_signals.index, buy_signals['Close'], marker='^', color='green', s=100, label='매수', zorder=5)
ax1.scatter(sell_signals.index, sell_signals['Close'], marker='v', color='red', s=100, label='매도', zorder=5)
ax1.set_ylabel('가격 ($)', fontsize=12)
ax1.set_title(f'{TICKER} MACD 백테스팅', fontsize=16, fontweight='bold')
ax1.legend(loc='upper left')
ax1.grid(True, alpha=0.3)

# 2. MACD Oscillator
colors = ['red' if val < 0 else 'green' for val in df['Oscillator']]
ax2.bar(df.index, df['Oscillator'], color=colors, alpha=0.7)
ax2.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
ax2.set_ylabel('MACD', fontsize=12)
ax2.grid(True, alpha=0.3)

# 3. 포트폴리오 가치
ax3.plot(df.index, df['Portfolio_Value'], label='전략 포트폴리오', linewidth=2, color='purple')
ax3.fill_between(df.index, df['Portfolio_Value'], INITIAL_CAPITAL, alpha=0.3, color='purple')
ax3.axhline(y=INITIAL_CAPITAL, color='gray', linestyle='--', label=f'초기 자본 (${INITIAL_CAPITAL:,})')
ax3.set_ylabel('포트폴리오 가치 ($)', fontsize=12)
ax3.set_xlabel('날짜', fontsize=12)
ax3.legend(loc='upper left')
ax3.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# 저장 옵션
save_chart = input("\n차트를 저장하시겠습니까? (y/n): ").lower()
if save_chart == 'y':
    filename = f'preview/{TICKER}_MACD_backtest_{datetime.now().strftime("%Y%m%d_%H%M%S")}.png'
    fig.savefig(filename, dpi=300, bbox_inches='tight')
    print(f"✅ 차트 저장됨: {filename}")

## 📝 7. 거래 내역

In [ ]:
# 거래 내역 추출
trades = df[df['Signal'] != 0][['Close', 'Signal', 'MA1', 'MA2']].copy()
trades['거래유형'] = trades['Signal'].map({1: '매수 🟢', -1: '매도 🔴'})
trades = trades.rename(columns={
    'Close': '가격',
    'MA1': f'MA{MA1}',
    'MA2': f'MA{MA2}'
})

print(f"\n총 {len(trades)}건의 거래:")
display(trades[['거래유형', '가격', f'MA{MA1}', f'MA{MA2}']])

## 💾 8. 결과 저장 (선택사항)

In [ ]:
# CSV로 저장
save_csv = input("결과를 CSV로 저장하시겠습니까? (y/n): ").lower()
if save_csv == 'y':
    filename = f'data/{TICKER}_backtest_{datetime.now().strftime("%Y%m%d_%H%M%S")}.csv'
    df.to_csv(filename)
    print(f"✅ 데이터 저장됨: {filename}")

print("\n🎉 백테스팅 완료!")
print("\n💡 다른 티커나 파라미터를 시도하려면:")
print("   1. 셀 3번(파라미터)으로 돌아가기")
print("   2. 값 수정")
print("   3. 셀 3번부터 다시 실행")

---

## 🔒 프라이버시 안내

- ✅ 모든 계산은 **로컬 컴퓨터**에서만 실행됩니다
- ✅ Yahoo Finance에서 공개 데이터만 다운로드합니다
- ✅ 귀하의 백테스팅 결과는 **외부로 전송되지 않습니다**
- ✅ 저장된 파일은 모두 로컬에만 존재합니다

---

## 📚 다음 단계

1. **파라미터 최적화**: 다양한 MA 조합 시도
2. **다른 티커 테스트**: NVDA, TSLA, MSFT 등
3. **기간 변경**: 다른 시장 상황 테스트
4. **다른 전략 시도**: Pair Trading, Bollinger Bands 등

**Happy Trading! 📈**